# Overview

This should work with Ghidra 10.3 (from https://github.com/NationalSecurityAgency/ghidra/releases/tag/Ghidra_10.3_build). In theory it could work for other 
versions, but the decompiler variables may not line up with my test data otherwise.

To get up and running:
1. Create a Ghidra project and import the included binary file `0.fighter`. 
2. Set the variable `project_folder` in the cell below to a pathlib object to this folder
3. Create a new python venv (3.8.10 or later) with `python -m venv <path_to_my_venv>`
4. Select this virtual environment as the interpreter for this Jupyter notebook
5. Run the notebook, and you should be able to see everything work and the "TODO" printouts for your code

In [18]:
!pip install pandas git+ssh://git@github.com/lasserre/astlib.git@dylan-dev

  Cloning ssh://****@github.com/lasserre/astlib.git (to revision dylan-dev) to /tmp/pip-req-build-8m2fjdn4
  Running command git clone -q 'ssh://****@github.com/lasserre/astlib.git' /tmp/pip-req-build-8m2fjdn4
  Running command git checkout -b dylan-dev --track origin/dylan-dev
  Switched to a new branch 'dylan-dev'
  Branch 'dylan-dev' set up to track remote branch 'dylan-dev' from 'origin'.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
  Cloning ssh://****@github.com/lasserre/wildebeest.git (to revision eb57039) to /tmp/pip-req-build-lgg1i1vs
  Running command git clone -q 'ssh://****@github.com/lasserre/wildebeest.git' /tmp/pip-req-build-lgg1i1vs
  Running command git checkout -q eb57039
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
  Created wheel for astlib: filename=astlib-0.0.1-py3-none-any.whl size=77444 sha256=c4c2ab19a

## Set these variables to match your Ghidra project

In [19]:
from pathlib import Path
import pandas as pd

folder_path = '/run1.gcc-O0.astera'     # CHANGE TO MATCH YOUR PROJECT - I expect yours will be '/' or ''
program_name = '0.fighter'              # this should be fine as-is
readonly = False

#---------------------------------------------
# point this to the Ghidra project you created
#---------------------------------------------
project_folder = Path.home()/'ghidra_projects'/'astera.gpr'

Have to import/start `pyhidra` before importing anything from `ghidra`

In [20]:
import pyhidra
pyhidra.start()

## Open Ghidra Project

In [21]:
import typing
if typing.TYPE_CHECKING:
    import ghidra
    from ghidra.ghidra_builtins import *

from ghidralib.datatypes import to_varlib_dtype
from varlib import StructDatabase

import ghidra
from ghidra.base.project import GhidraProject

gproj = GhidraProject.openProject(project_folder.parent, project_folder.stem, False)
prog = gproj.openProgram(folder_path, program_name, readonly)
sdb = StructDatabase.from_json('0.fighter.debug.sdb')

ModuleNotFoundError: No module named 'tqdm'

# TODO: Implement `GhidraRetyper` Functionality
See comments below :)

In [ ]:
# ghidra imports
from ghidra.program.model.listing import Program

# my stuff (astlib)
from varlib import datatype, StructDatabase

###################################################################
# STUB CLASS/FUNCTIONS (aka what I need you to implement)
# -----------------------------------------------------------------
# This is a specific example of the general idea I'm looking for...lol
#
# What I mean is, we can totally tweak the API - this is just my best
# guess at a concrete example of what I need
###################################################################

class GhidraRetyper:
    def __init__(self, program:Program, reference_db:StructDatabase) -> None:
        # NOTE: I think this is all the Ghidra context you'll need
        # to access database things but add more if you need to
        self.program = program

        # right now the db I'm handing you has everything, but
        # the only guarantee is that it has definitions for all of
        # the structure/union types I actually ask you to apply
        # (and their dependencies) so it may be a subset of
        # Ghidra's full data type archive in general
        self.reference_db = reference_db

    def define_all_reference_types(self, overwrite_existing:bool=False):

        # this is where you may need to perform a topological sort
        # or w/e to handle dependencies (structs within structs)
        # Jacob did this recently...may be good to ask him about it
        print(f'DYLAN TODO: make sure this works for nested structure/union types...')

        # Caleb's naive implementation:
        for sid, stype in self.reference_db.structs_by_id:
            self.define_struct_type(stype, overwrite_existing)

        for uid, utype in self.reference_db.unions_by_id:
            self.define_union_type(utype, overwrite_existing)

    def define_struct_type(self, stype:datatype.StructType, overwrite_existing:bool=False):
        # define the structure using its given name and layout
        # --> IGNORE THE SID (stype.sid). Ghidra will assign its own ID and I will
        #     probably re-export the data type archive and remap the ids outside of this
        #     Regardless, I won't assume the sids are still valid once this completes.
        #     I will only assume the structure with the given name is defined in the database

        # also...if Ghidra wants you to pick a path for the data types (that show up in
        # the data type manager tree on the left), we can just use some hardcoded path
        # for all type we define for now (e.g. "/GhidraRetyper")

        # if overwrite_existing then blow away an existing struct with the same name
        print(f'DYLAN TODO: define structure type {stype.name} in Ghidra')

    def define_union_type(self, utype:datatype.UnionType, overwrite_existing:bool=False):
        # same as above, but unions...
        print(f'DYLAN TODO: define union type {utype.name} in Ghidra')

    def set_localvar_type(self, func_addr:int, local_name:str, dtype:datatype.DataType):
        print(f'DYLAN TODO: set decompiler local {local_name} in {func_addr:#x} to type {dtype}')

    def set_globalvar_type(self, global_name:int, global_type:datatype.DataType):
        # do this last: I don't have data for globals right now and we may not need them
        # ...but, while you're doing the others if this is straightforward you can add
        # support for globals too
        pass

    def set_param_type(self, func_addr:int, param_name:str, dtype:datatype.DataType):
        print(f'DYLAN TODO: set decompiler parameter {param_name} in {func_addr:#x} to type {dtype}')

    def set_return_type(self, func_addr:int, dtype:datatype.DataType):
        print(f'DYLAN TODO: set decompiler return type for function {func_addr:#x} to type {dtype}')

: 

# Driver Code
Here are my test cases and a good example of how I will use your class. The one I have set is a single
function for starters...you can change the function 

In [ ]:
# --------------------------------
# Single test function "init_game" - you can set this to different function names to test individual cases
test_func_name = 'init_game'
only_test_func = True       # flip this to false to run against ALL variables for a more thorough test case
# --------------------------------

: 

In [ ]:
# read in my test data
funcs_df = pd.read_csv('functions.csv')
locals_df = pd.read_csv('locals.csv')

if only_test_func:
    addr = funcs_df[funcs_df.FunctionName_Debug==test_func_name].FunctionStart.iloc[0]
    print(f'Using test function {test_func_name} @ {addr:#x}')
    func_locals = locals_df[locals_df.FunctionStart==addr]
    df = func_locals
else:
    df = locals_df
    print(f'Running across all functions')

: 

In [ ]:
from varlib.datatype import datatype_from_dict
import json

# instantiate your class and apply types
retyper = GhidraRetyper(prog, sdb)

for i in range(len(df)):
    x = df.iloc[i]
    if x.TypeCategory_Debug == 'COMP':
        continue
    varlib_dt = datatype_from_dict(json.loads(x.TypeJson_Debug), sdb)

    # start with local vars. you're welcome to branch out to params/globals
    # but I only have test data for locals right now
    retyper.set_localvar_type(addr, x.Name_Strip, varlib_dt)


: 